In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

In [2]:
employees = pd.read_csv("../data/employees_raw.csv")
regions = pd.read_csv("../data/region_benefit_profiles.csv")

In [3]:
employees.rename(
    columns={"qemployee_id": "employee_id"},
    inplace=True
)                                             #column rename - qemployee_id -- employee_id

In [4]:
employees.columns

Index(['employee_id', 'age', 'gender', 'marital_status', 'salary',
       'employment_type', 'region', 'has_dependents', 'tenure_years',
       'enrolled', 'application_date', 'last_contact_date',
       'last_contact_channel', 'plan_tier_requested', 'broker_channel',
       'prior_year_enrolled', 'legacy_propensity_score', 'outreach_notes'],
      dtype='object')

In [5]:
employees["employee_id"].duplicated().sum()

np.int64(8)

See we will delete all the duplicate rows (i.e the rows having same employee_id)

Reason : We could have kept one of them if the last_contacted_date would have been different which would have detected that there might be some updation problem (also, we would have kept the data with latest last_contacted_date but as There's only difference in enrolled column and performance rank, so we need to delete all the duplicates for better practice

In [6]:
duplicate_ids = employees[
    employees["employee_id"].duplicated(keep=False)
]["employee_id"]

employees = employees[
    ~employees["employee_id"].isin(duplicate_ids)
].copy()

In [7]:
employees["employee_id"].duplicated().sum()

np.int64(0)

In [8]:
employees.shape

(9992, 18)

In [9]:
employees["last_contact_channel"].value_counts(dropna=False)

last_contact_channel
NaN       1181
EMAIL     1162
email     1130
e-mail    1109
Email     1098
PHONE      622
Phone      617
phone      602
Call       587
Text       500
sms        490
SMS        488
none       406
Name: count, dtype: int64

In [10]:
employees["last_contact_channel"] = (
    employees["last_contact_channel"]
    .str.lower()
    .str.strip()
)

In [11]:
channel_mapping = {
    "email": "Email",
    "e-mail": "Email",

    "phone": "Phone",
    "call": "Phone",

    "sms": "SMS",
    "text": "SMS",

    "none": "None"
}

employees["last_contact_channel"] = (
    employees["last_contact_channel"]
    .replace(channel_mapping)
)

In [12]:
employees["last_contact_channel"].value_counts(dropna=False)

last_contact_channel
Email    4499
Phone    2428
SMS      1478
NaN      1181
None      406
Name: count, dtype: int64

In [13]:
employees["plan_tier_requested"].value_counts(dropna=False)

plan_tier_requested
STANDARD        1322
Standard        1315
Silver          1302
standard        1282
silver plan     1263
Bronze           532
NaN              524
BASIC            512
basic            508
Basic            490
premium plan     201
Premium          155
PREMIUM          154
gold             152
Gold Plan        142
Gold             138
Name: count, dtype: int64

In [14]:
employees["plan_tier_requested"] = (
    employees["plan_tier_requested"]
    .str.lower()
    .str.strip()
)

In [15]:
plan_mapping = {
    "basic": "Basic",

    "standard": "Standard",

    "silver": "Silver",
    "silver plan": "Silver",

    "premium": "Premium",
    "premium plan": "Premium",

    "gold": "Gold",
    "gold plan": "Gold",

    "bronze": "Bronze"
}

employees["plan_tier_requested"] = (
    employees["plan_tier_requested"]
    .replace(plan_mapping)
)

In [16]:
employees["plan_tier_requested"].value_counts(dropna=False)

plan_tier_requested
Standard    3919
Silver      2565
Basic       1510
Bronze       532
NaN          524
Premium      510
Gold         432
Name: count, dtype: int64

In [17]:
regions["state_mandate_level"]

0    High
1     low
2     MED
3     Low
Name: state_mandate_level, dtype: object

In [18]:
regions["state_mandate_level"] = (
    regions["state_mandate_level"]
    .str.lower()
    .str.strip()
)

In [19]:
state_mapping = {
    "low": "Low",
    "med": "Medium",
    "high": "High"
}

regions["state_mandate_level"] = (
    regions["state_mandate_level"]
    .replace(state_mapping)
)

In [20]:
regions["state_mandate_level"].value_counts()

state_mandate_level
Low       2
High      1
Medium    1
Name: count, dtype: int64

Let's manage date now

In [21]:
employees[["application_date", "last_contact_date"]].head(20)

,application_date,last_contact_date
0,2024-04-14,2024-03-30
1,2024-05-09,2024-04-18
2,22/06/2024,2024-06-09
3,2024-08-16,2024-07-29
4,13/05/2024,2024-05-06
5,14/01/2024,2024-01-12
6,2024-07-23,2024-07-13
7,20-Aug-2024,2024-08-01
8,NaN,2024-05-16
9,2024-06-18,2024-05-26


In [22]:
employees["application_date"] = pd.to_datetime(
    employees["application_date"],
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

employees["last_contact_date"] = pd.to_datetime(
    employees["last_contact_date"],
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

In [23]:
employees.info()

<class 'pandas.core.frame.DataFrame'>
Index: 9992 entries, 0 to 10007
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   employee_id              9992 non-null   int64         
 1   age                      9992 non-null   int64         
 2   gender                   9992 non-null   object        
 3   marital_status           9992 non-null   object        
 4   salary                   9992 non-null   float64       
 5   employment_type          9992 non-null   object        
 6   region                   9992 non-null   object        
 7   has_dependents           9992 non-null   object        
 8   tenure_years             9992 non-null   float64       
 9   enrolled                 9992 non-null   int64         
 10  application_date         9273 non-null   datetime64[ns]
 11  last_contact_date        9992 non-null   datetime64[ns]
 12  last_contact_channel     8811 non-null

Let's find impossible dates i.e last_contact_date > application_date

In [24]:
invalid_dates = employees[
    employees["last_contact_date"] >
    employees["application_date"]
]

invalid_dates.shape

(61, 18)

In [25]:
invalid_dates[
    ["employee_id",
     "application_date",
     "last_contact_date"]
].head(20)

,employee_id,application_date,last_contact_date
59,18661,2024-03-23,2024-03-28
513,18287,2024-03-06,2024-03-08
882,13294,2024-03-11,2024-03-15
1020,15439,2024-09-18,2024-09-20
1120,12726,2024-03-16,2024-03-24
1162,13675,2024-05-20,2024-05-27
1219,16644,2024-04-21,2024-04-28
1226,10064,2024-01-16,2024-01-19
1552,18876,2024-09-29,2024-10-08
1615,13503,2024-01-28,2024-02-06


Identified 50 records where last_contact_date occurs after application_date. While this may indicate a chronological inconsistency, it could also represent legitimate follow-up communication after an application was submitted. These records are retained pending further business rules or domain clarification.

In [26]:
#lets go to missing values now--

In [27]:
missing = employees.isnull().sum().sort_values(ascending=False)

missing

outreach_notes             1453
last_contact_channel       1181
legacy_propensity_score     899
application_date            719
plan_tier_requested         524
broker_channel              468
age                           0
prior_year_enrolled           0
last_contact_date             0
employee_id                   0
tenure_years                  0
has_dependents                0
region                        0
employment_type               0
salary                        0
marital_status                0
gender                        0
enrolled                      0
dtype: int64

In [28]:
missing_percentage = (
    employees.isnull().sum() / len(employees)
) * 100

pd.DataFrame({
    "Missing Values": missing,
    "Percentage": missing_percentage
}).sort_values(by="Percentage", ascending=False)

,Missing Values,Percentage
outreach_notes,1453,14.541633
last_contact_channel,1181,11.819456
legacy_propensity_score,899,8.997198
application_date,719,7.195757
plan_tier_requested,524,5.244195
broker_channel,468,4.683747
age,0,0.000000
marital_status,0,0.000000
salary,0,0.000000
region,0,0.000000


In [29]:
missing = employees.isnull().sum()

missing_percentage = (
    missing / len(employees) * 100
).round(2)

missing_summary = pd.DataFrame({
    "Missing Values": missing,
    "Percentage": missing_percentage
})

missing_summary = missing_summary[
    missing_summary["Missing Values"] > 0
].sort_values(
    by="Percentage",
    ascending=False
)

missing_summary

,Missing Values,Percentage
outreach_notes,1453,14.54
last_contact_channel,1181,11.82
legacy_propensity_score,899,9.00
application_date,719,7.20
plan_tier_requested,524,5.24
broker_channel,468,4.68


In [30]:
employees["broker_channel"].value_counts(dropna=False)

broker_channel
Direct                3533
Third-Party           3006
Employer-Sponsored    2985
NaN                    468
Name: count, dtype: int64

In [31]:
employees["plan_tier_requested"].value_counts(dropna=False)

plan_tier_requested
Standard    3919
Silver      2565
Basic       1510
Bronze       532
NaN          524
Premium      510
Gold         432
Name: count, dtype: int64

HANDLING MISSING VALUES

In [32]:
employees["plan_tier_requested"] = (
    employees["plan_tier_requested"]
    .fillna("Unknown")
)

In [33]:
employees["last_contact_channel"] = (
    employees["last_contact_channel"]
    .fillna("Unknown")
)

In [34]:
employees["broker_channel"] = (
    employees["broker_channel"]
    .fillna("Unknown")
)

In [35]:
employees["prior_year_enrolled"].value_counts(dropna=False)

prior_year_enrolled
-1    4042
 1    3571
 0    2379
Name: count, dtype: int64

In [36]:
employees["prior_year_enrolled"] = (
    employees["prior_year_enrolled"]
    .replace({
        -1: "New Hire",
         0: "No",
         1: "Yes"
    })
)

In [37]:
employees["prior_year_enrolled"].value_counts()

prior_year_enrolled
New Hire    4042
Yes         3571
No          2379
Name: count, dtype: int64

In [38]:
employees.isnull().sum()

employee_id                   0
age                           0
gender                        0
marital_status                0
salary                        0
employment_type               0
region                        0
has_dependents                0
tenure_years                  0
enrolled                      0
application_date            719
last_contact_date             0
last_contact_channel          0
plan_tier_requested           0
broker_channel                0
prior_year_enrolled           0
legacy_propensity_score     899
outreach_notes             1453
dtype: int64

In [39]:
employees.head(20)

,employee_id,age,gender,marital_status,salary,employment_type,region,has_dependents,tenure_years,enrolled,application_date,last_contact_date,last_contact_channel,plan_tier_requested,broker_channel,prior_year_enrolled,legacy_propensity_score,outreach_notes
0,12324,28,Male,Divorced,44047.60,Full-time,West,No,1.4,0,2024-04-14,2024-03-30,SMS,Basic,Employer-Sponsored,No,0.104,Spouse covered elsewhere
1,17825,23,Male,Married,80111.28,Full-time,Midwest,Yes,0.5,1,2024-05-09,2024-04-18,Phone,Standard,Direct,New Hire,0.874,Requested more time
2,15200,39,Male,Married,69855.65,Full-time,South,Yes,3.4,1,2024-06-22,2024-06-09,None,Silver,Direct,No,0.870,No response to first outreach
3,16690,31,Female,Married,91567.33,Full-time,South,No,1.2,1,2024-08-16,2024-07-29,Email,Premium,Employer-Sponsored,Yes,0.894,Requested more time
4,17465,42,Female,Single,59861.68,Contract,Midwest,Yes,23.8,0,2024-05-13,2024-05-06,Email,Standard,Employer-Sponsored,New Hire,0.163,Requested more time
5,17727,52,Female,Married,51330.25,Contract,Northeast,No,0.8,0,2024-01-14,2024-01-12,Email,Basic,Direct,New Hire,0.352,Attended benefits webinar
6,13582,59,Female,Widowed,74884.52,Full-time,Northeast,No,0.7,1,2024-07-23,2024-07-13,SMS,Silver,Third-Party,New Hire,0.849,Attended benefits webinar
7,17068,61,Male,Single,73525.85,Full-time,Midwest,Yes,1.4,1,2024-08-20,2024-08-01,Email,Silver,Third-Party,New Hire,0.840,Requested more time
8,16183,38,Male,Single,70490.72,Full-time,West,No,2.3,1,NaT,2024-05-16,Phone,Standard,Employer-Sponsored,New Hire,NaN,Follow-up scheduled
9,16268,39,Female,Single,50620.37,Full-time,West,Yes,5.6,1,2024-06-18,2024-05-26,Email,Basic,Direct,Yes,0.847,No response to first outreach


In [40]:
import pandas as pd
print(pd.__version__)

2.3.3


In [41]:
employees.isnull().sum()

employee_id                   0
age                           0
gender                        0
marital_status                0
salary                        0
employment_type               0
region                        0
has_dependents                0
tenure_years                  0
enrolled                      0
application_date            719
last_contact_date             0
last_contact_channel          0
plan_tier_requested           0
broker_channel                0
prior_year_enrolled           0
legacy_propensity_score     899
outreach_notes             1453
dtype: int64

In [42]:
invalid_dates = employees[
    employees["last_contact_date"] > employees["application_date"]
]

print(len(invalid_dates))

61


In [43]:
invalid_dates[
    [
        "employee_id",
        "last_contact_date",
        "application_date",
        "enrolled",
        "outreach_notes"
    ]
].head(15)

,employee_id,last_contact_date,application_date,enrolled,outreach_notes
59,18661,2024-03-28,2024-03-23,0,Requested more time
513,18287,2024-03-08,2024-03-06,1,Requested more time
882,13294,2024-03-15,2024-03-11,1,Attended benefits webinar
1020,15439,2024-09-20,2024-09-18,1,Declined - cost concern
1120,12726,2024-03-24,2024-03-16,0,Requested more time
1162,13675,2024-05-27,2024-05-20,1,Declined - cost concern
1219,16644,2024-04-28,2024-04-21,1,Spouse covered elsewhere
1226,10064,2024-01-19,2024-01-16,0,Spouse covered elsewhere
1552,18876,2024-10-08,2024-09-29,1,Follow-up scheduled
1615,13503,2024-02-06,2024-01-28,1,Declined - cost concern


IN THE ABOVE TABLE IT'S CLEARLY VISIBLE THAT INSPITE OF THE COMMENTS "Declined - cost concern", "Spouse covered elsewhere"; THE enrolled COLUMN SHOWS STILL 1

THIS PROVES THAT outreach_notes COLUMN VALUES ARE NOISY VALUES

SO, LET'S DROP THAT COLUMN!

In [44]:
employees = employees.drop(columns=["outreach_notes"])

In [45]:
employees["region"].unique()

array(['West', 'Midwest', 'South', 'Northeast'], dtype=object)

In [46]:
regions["region"].unique()

array(['Midwest', 'Northeast', 'South', 'West'], dtype=object)

In [47]:
merged = employees.merge(
    regions,
    on="region",
    how="left"
)

In [48]:
merged.shape

(9992, 25)

In [49]:
employees.shape

(9992, 17)

In [50]:
regions.head()


,region,n_employees_region,hist_enrollment_rate_region,avg_salary_region,avg_premium_cost_usd,benefits_broker_rating,hr_outreach_capacity,open_enrollment_window_days,state_mandate_level
0,Midwest,2488,0.617,64921.46,595,4.4,469,18,High
1,Northeast,2506,0.612,65008.32,455,3.1,151,38,Low
2,South,2424,0.628,65200.49,481,3.3,324,17,Medium
3,West,2582,0.613,65007.07,574,4.7,437,28,Low


In [51]:
regions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 9 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   region                       4 non-null      object 
 1   n_employees_region           4 non-null      int64  
 2   hist_enrollment_rate_region  4 non-null      float64
 3   avg_salary_region            4 non-null      float64
 4   avg_premium_cost_usd         4 non-null      int64  
 5   benefits_broker_rating       4 non-null      float64
 6   hr_outreach_capacity         4 non-null      int64  
 7   open_enrollment_window_days  4 non-null      int64  
 8   state_mandate_level          4 non-null      object 
dtypes: float64(3), int64(4), object(2)
memory usage: 420.0+ bytes


In [52]:
merged.to_csv(
    "../data/employees_final.csv",
    index=False
)